# Сбор датасета кошек и собак из Яндекс Картинок

Цель — получить изображения двух классов, отфильтровать неподходящие файлы и сохранить происхождение каждого принятого изображения в манифесте. Ниже проверяются ключевые части реализации из `main.py`; повторный сбор запускается только в последней ячейке.

## Подключение реализации

В notebook используются функции из основного скрипта, поэтому проверяем именно тот код, который выполняет сбор.

In [1]:
from pathlib import Path
import io
import json

from PIL import Image

import main

PROJECT_DIR = Path.cwd()
assert (PROJECT_DIR / 'main.py').exists(), 'Запустите notebook из каталога проекта'
print(f'Рабочий каталог: {PROJECT_DIR}')

Рабочий каталог: f:\NTSE\lab01


## Настройки выборки

Для каждого класса предусмотрено несколько запросов. Поиск запрашивает крупные изображения, а `MIN_SIDE` дополнительно отсеивает файлы, у которых хотя бы одна сторона меньше 600 px.

In [2]:
print('Целевое количество на класс:', main.TARGET_IMAGES)
print('Минимальная сторона:', main.MIN_SIDE, 'px')
for class_name, queries in main.SEARCH_QUERIES.items():
    print(f'{class_name}: {queries}')

Целевое количество на класс: 1200
Минимальная сторона: 600 px
cat: ['cat', 'кот', 'кошка', 'домашняя кошка']
dog: ['dog', 'собака', 'пёс']


## Извлечение ссылок на оригиналы

Функция `extract_image_urls` разбирает данные результата поиска и берёт `img_href`/`origUrl`, то есть ссылку на исходное изображение, а не миниатюру. Проверим её на небольшом HTML-примере без обращения к сети.

In [3]:
sample_html = '''
<div class="serp-item" data-bem='{&quot;serp-item&quot;: {&quot;img_href&quot;: &quot;//images.example.org/cat.jpg&quot;}}'></div>
<div class="serp-item" data-bem='{&quot;serp-item&quot;: {&quot;img_href&quot;: &quot;//images.example.org/cat.jpg&quot;}}'></div>
'''
urls = main.extract_image_urls(sample_html)
assert urls == ['https://images.example.org/cat.jpg']
urls

['https://images.example.org/cat.jpg']

## Проверка и нормализация изображения

До сохранения байты открываются Pillow, учитывается EXIF-поворот, проверяется размер, затем файл приводится к JPEG/RGB. Это исключает повреждённые, слишком маленькие и неподходящие по формату изображения.

In [4]:
# Создаём RGBA-изображение в памяти: функция должна вернуть корректный JPEG.
rgba = Image.new('RGBA', (800, 650), (30, 120, 220, 128))
buffer = io.BytesIO()
rgba.save(buffer, format='PNG')

converted = main.convert_to_jpeg(buffer.getvalue())
assert converted is not None
jpeg_bytes, dimensions = converted
with Image.open(io.BytesIO(jpeg_bytes)) as image:
    assert image.format == 'JPEG' and image.mode == 'RGB'

print(f'Размер исходного изображения: {dimensions}; JPEG: {len(jpeg_bytes):,} байт')

Размер исходного изображения: (800, 650); JPEG: 3,363 байт


## Текущее состояние датасета

Манифест хранит класс, локальный файл, URL источника, SHA-256, размер, запрос и страницу выдачи. При новом запуске URL и хэши уже принятых файлов загружаются через `load_seen`, поэтому дубликаты не сохраняются.

In [5]:
seen_urls, seen_hashes = main.load_seen()
summary = {}
for class_name in main.SEARCH_QUERIES:
    class_dir = main.DATASET_DIR / class_name
    summary[class_name] = len(list(class_dir.glob('*.jpg')))

print('Файлов по классам:', summary)
print('Уникальных URL в манифесте:', len(seen_urls))
print('Уникальных SHA-256:', len(seen_hashes))
assert all(count <= main.TARGET_IMAGES for count in summary.values())

Файлов по классам: {'cat': 1200, 'dog': 1107}
Уникальных URL в манифесте: 2310
Уникальных SHA-256: 2310


In [7]:
records = [json.loads(line) for line in main.MANIFEST_PATH.read_text(encoding='utf-8').splitlines() if line]
assert all(record['width'] >= main.MIN_SIDE and record['height'] >= main.MIN_SIDE for record in records)
assert len({record['sha256'] for record in records}) == len(records), 'В манифесте есть дубли по SHA-256'

for class_name in main.SEARCH_QUERIES:
    class_records = [record for record in records if record['class'] == class_name]
    print(f'{class_name}: {len(class_records)} записей, пример размера: {class_records[0]["width"]}×{class_records[0]["height"]}')

cat: 1200 записей, пример размера: 2560×1440
dog: 1110 записей, пример размера: 2048×2560
